# Repsol–IE Sustainability Challenge Final Notebook (Using cleanDatav3.csv)

This notebook integrates data from earlier stages (EDA, cleaning, and feature engineering) to build optimal models for predicting the maximum possible solar generation. In previous runs we achieved a training MAE around 9.4, but our evaluation MAE on September data is near 11.0. 

Further refinements can include:
- Refining feature engineering with additional polynomial or domain-driven transformations
- Adding regularization (e.g., tuning `reg_lambda`, `reg_alpha` in XGBoost or exploring CatBoost)
- Incorporating external data (holidays, day-of-week indicators, or enhanced weather data)
- Iteratively monitoring the MAE on the September test set and refining the model
- Integrating battery optimization and CO₂ reduction logic once the solar forecast is optimized

This notebook also exports predictions to CSV for submission via the official challenge forms.

Good luck, and keep iterating to lower that MAE!

In [1]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
#from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV

import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

plt.style.use('default')
pd.set_option('display.max_columns', None)

print('Libraries imported successfully!')

Libraries imported successfully!


## 1) Data Loading & Initial Cleaning

Load the cleaned dataset **cleanDatav3.csv**. Then drop any unwanted columns (e.g., `GETAFE_SOLAR` if present).

In [2]:
# Load cleaned data from cleanDatav3.csv
df = pd.read_csv('cleanDatav0.csv')

# Print basic information
print('Data shape:', df.shape)
print('Columns:', df.columns.tolist())
display(df.head())

# Drop unwanted column 'GETAFE_SOLAR' if it exists
if 'GETAFE_SOLAR' in df.columns:
    df.drop(columns=['GETAFE_SOLAR'], inplace=True)
    print("Dropped column 'GETAFE_SOLAR'.")
else:
    print("Column 'GETAFE_SOLAR' not found.")

Data shape: (9798, 30)
Columns: ['TIMESTAMP', 'W_10uheightAboveGround_10', 'W_10vheightAboveGround_10', 'W_2rheightAboveGround_2', 'W_2shheightAboveGround_2', 'W_2theightAboveGround_2', 'W_SUNSDsurface_0', 'W_aptmpheightAboveGround_2', 'W_dlwrfsurface_0', 'W_dswrfsurface_0', 'W_gustsurface_0', 'W_msletmeanSea_0', 'W_presheightAboveGround_80', 'W_pwatatmosphereSingleLayer_0', 'W_qheightAboveGround_80', 'W_spsurface_0', 'W_tccatmosphere_0', 'W_theightAboveGround_80', 'W_tmaxheightAboveGround_2', 'W_tminheightAboveGround_2', 'W_tozneatmosphereSingleLayer_0', 'W_tpsurface_0', 'W_tsurface_0', 'W_uheightAboveGround_80', 'W_vheightAboveGround_80', 'SG_TOTAL_KWH_ENERGIA', 'GI_TOTAL_KWH_ENERGIA', 'GETAFE_SOLAR', 'hour', 'month']


,TIMESTAMP,W_10uheightAboveGround_10,W_10vheightAboveGround_10,W_2rheightAboveGround_2,W_2shheightAboveGround_2,W_2theightAboveGround_2,W_SUNSDsurface_0,W_aptmpheightAboveGround_2,W_dlwrfsurface_0,W_dswrfsurface_0,W_gustsurface_0,W_msletmeanSea_0,W_presheightAboveGround_80,W_pwatatmosphereSingleLayer_0,W_qheightAboveGround_80,W_spsurface_0,W_tccatmosphere_0,W_theightAboveGround_80,W_tmaxheightAboveGround_2,W_tminheightAboveGround_2,W_tozneatmosphereSingleLayer_0,W_tpsurface_0,W_tsurface_0,W_uheightAboveGround_80,W_vheightAboveGround_80,SG_TOTAL_KWH_ENERGIA,GI_TOTAL_KWH_ENERGIA,GETAFE_SOLAR,hour,month
0,2023-07-25 01:00:00,2.766361,-1.444303,24.500379,0.003757,187.853340,0.0,187.876484,202.352603,0.0,4.249933,64996.490009,59880.745214,9.822742,0.003744,60436.914231,3.078582,188.549438,188.630032,187.838371,203.820773,0.0,186.691889,3.662784,-2.716726,0.0,110.063507,0.0,1,7
1,2023-07-25 02:00:00,2.622140,-1.138907,27.707234,0.003929,187.157423,0.0,187.112796,201.007227,0.0,3.729249,64984.508393,59857.756066,9.944934,0.003907,60416.875391,1.218605,187.877266,188.632577,187.132862,202.629868,0.0,185.865653,3.231792,-2.302657,0.0,110.063507,0.0,2,7
2,2023-07-25 03:00:00,2.233882,-1.599228,29.118251,0.004049,186.964910,0.0,186.940060,200.290243,0.0,3.345747,64995.131103,59858.209035,9.808392,0.004021,60416.420418,0.000000,187.673800,188.613919,186.876314,202.231360,0.0,185.675285,2.794408,-2.799250,0.0,110.063507,0.0,3,7
3,2023-07-25 04:00:00,1.754841,-1.310309,30.080308,0.004053,186.580839,0.0,186.574873,199.870907,0.0,2.182487,65011.782702,59864.594686,9.901714,0.004017,60423.770131,0.000000,187.501040,188.607593,186.555628,201.784076,0.0,185.094536,1.830703,-2.208902,0.0,54.867619,0.0,4,7
4,2023-07-25 05:00:00,0.993990,-0.987727,31.555461,0.004049,186.064422,0.0,186.081427,199.466431,0.0,1.544254,65051.054661,59894.342283,10.048406,0.003987,60453.154951,0.000000,187.399072,188.607593,186.042532,200.422649,0.0,184.157314,0.748354,-1.544372,0.0,54.867619,5.0,5,7


Dropped column 'GETAFE_SOLAR'.


## 2) Feature Engineering & VIF Checking

We create additional features, including time-based features and polynomial transformations of key meteorological variables. We also check for multicollinearity using the Variance Inflation Factor (VIF).

In [3]:
# Set target column by renaming if necessary
if 'pv_generation' not in df.columns and 'SG_TOTAL_KWH_ENERGIA' in df.columns:
    df.rename(columns={'SG_TOTAL_KWH_ENERGIA': 'pv_generation'}, inplace=True)

# Create datetime features if not already present
if 'datetime' not in df.columns and 'TIMESTAMP' in df.columns:
    df.rename(columns={'TIMESTAMP': 'datetime'}, inplace=True)

df['datetime'] = pd.to_datetime(df['datetime'])
df['hour'] = df['datetime'].dt.hour
df['dayofyear'] = df['datetime'].dt.dayofyear

# Create meteorological features from 'W_dswrfsurface_0' and 'W_tccatmosphere_0'
df['dswrfsurface_0'] = df['W_dswrfsurface_0']
df['tccatmosphere_0'] = df['W_tccatmosphere_0']

# Create polynomial features
df['dswrf_sq'] = df['dswrfsurface_0'] ** 2
df['dswrf_sqrt'] = np.sqrt(np.maximum(df['dswrfsurface_0'], 0))

# Define feature columns and target column
feature_cols = ['hour', 'dayofyear', 'dswrfsurface_0', 'dswrf_sq', 'dswrf_sqrt', 'tccatmosphere_0']
target_col = 'pv_generation'

# Check VIF to monitor multicollinearity
X_for_vif = df[feature_cols].dropna()
X_for_vif = sm.add_constant(X_for_vif)
vif_data = pd.DataFrame()
vif_data['Feature'] = X_for_vif.columns
vif_data['VIF'] = [variance_inflation_factor(X_for_vif.values, i) for i in range(X_for_vif.shape[1])]
print('VIF values:')
print(vif_data)

VIF values:
           Feature         VIF
0            const    9.307195
1             hour    1.151765
2        dayofyear    1.020084
3   dswrfsurface_0  144.621638
4         dswrf_sq   38.771980
5       dswrf_sqrt   47.273829
6  tccatmosphere_0    1.043282


## 3) Time-Based Split

We split the dataset using a time-based approach: data up to August 31, 2024 for training and September 2024 for prediction.

In [4]:
# Sort data by datetime
df.sort_values('datetime', inplace=True)

# Define cutoff dates
train_end = pd.to_datetime('2024-08-31')
sep_start = pd.to_datetime('2024-09-01')
sep_end = pd.to_datetime('2024-09-30 23:59:59')

df_train = df[df['datetime'] <= train_end].copy()
df_sep = df[(df['datetime'] >= sep_start) & (df['datetime'] <= sep_end)].copy()

print('Train data shape:', df_train.shape)
print('September data shape:', df_sep.shape)

Train data shape: (9055, 34)
September data shape: (720, 34)


In [5]:
df

,datetime,W_10uheightAboveGround_10,W_10vheightAboveGround_10,W_2rheightAboveGround_2,W_2shheightAboveGround_2,W_2theightAboveGround_2,W_SUNSDsurface_0,W_aptmpheightAboveGround_2,W_dlwrfsurface_0,W_dswrfsurface_0,W_gustsurface_0,W_msletmeanSea_0,W_presheightAboveGround_80,W_pwatatmosphereSingleLayer_0,W_qheightAboveGround_80,W_spsurface_0,W_tccatmosphere_0,W_theightAboveGround_80,W_tmaxheightAboveGround_2,W_tminheightAboveGround_2,W_tozneatmosphereSingleLayer_0,W_tpsurface_0,W_tsurface_0,W_uheightAboveGround_80,W_vheightAboveGround_80,pv_generation,GI_TOTAL_KWH_ENERGIA,hour,month,dayofyear,dswrfsurface_0,tccatmosphere_0,dswrf_sq,dswrf_sqrt
0,2023-07-25 01:00:00,2.766361,-1.444303,24.500379,0.003757,187.853340,0.0,187.876484,202.352603,0.0,4.249933,64996.490009,59880.745214,9.822742,0.003744,60436.914231,3.078582,188.549438,188.630032,187.838371,203.820773,0.0,186.691889,3.662784,-2.716726,0.0,110.063507,1,7,206,0.0,3.078582,0.0,0.0
1,2023-07-25 02:00:00,2.622140,-1.138907,27.707234,0.003929,187.157423,0.0,187.112796,201.007227,0.0,3.729249,64984.508393,59857.756066,9.944934,0.003907,60416.875391,1.218605,187.877266,188.632577,187.132862,202.629868,0.0,185.865653,3.231792,-2.302657,0.0,110.063507,2,7,206,0.0,1.218605,0.0,0.0
2,2023-07-25 03:00:00,2.233882,-1.599228,29.118251,0.004049,186.964910,0.0,186.940060,200.290243,0.0,3.345747,64995.131103,59858.209035,9.808392,0.004021,60416.420418,0.000000,187.673800,188.613919,186.876314,202.231360,0.0,185.675285,2.794408,-2.799250,0.0,110.063507,3,7,206,0.0,0.000000,0.0,0.0
3,2023-07-25 04:00:00,1.754841,-1.310309,30.080308,0.004053,186.580839,0.0,186.574873,199.870907,0.0,2.182487,65011.782702,59864.594686,9.901714,0.004017,60423.770131,0.000000,187.501040,188.607593,186.555628,201.784076,0.0,185.094536,1.830703,-2.208902,0.0,54.867619,4,7,206,0.0,0.000000,0.0,0.0
4,2023-07-25 05:00:00,0.993990,-0.987727,31.555461,0.004049,186.064422,0.0,186.081427,199.466431,0.0,1.544254,65051.054661,59894.342283,10.048406,0.003987,60453.154951,0.000000,187.399072,188.607593,186.042532,200.422649,0.0,184.157314,0.748354,-1.544372,0.0,54.867619,5,7,206,0.0,0.000000,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9793,2024-09-30 19:00:00,0.576858,0.917306,16.034279,0.002853,189.504855,0.0,189.494388,207.639218,0.0,1.156888,65198.894723,60152.965185,10.855934,0.002595,60704.317905,3.206856,190.997202,190.195472,189.504855,188.253297,0.0,187.760783,0.678673,1.126159,NaN,NaN,19,9,274,0.0,3.206856,0.0,0.0
9794,2024-09-30 20:00:00,0.364110,0.730518,17.060473,0.002899,189.012067,0.0,189.039681,207.025218,0.0,0.853867,65207.080223,60151.141286,10.494148,0.002584,60703.127360,3.206856,190.872060,190.195472,189.012067,189.485063,0.0,187.156930,0.449041,0.883448,NaN,NaN,20,9,274,0.0,3.206856,0.0,0.0
9795,2024-09-30 21:00:00,0.458659,0.709988,17.252884,0.002848,188.691398,0.0,188.741967,206.203974,0.0,0.919450,65206.078080,60141.284213,9.917611,0.002574,60693.797414,0.000000,190.656160,190.195472,188.691398,191.691640,0.0,186.660587,0.541201,0.864555,NaN,NaN,21,9,274,0.0,0.000000,0.0,0.0
9796,2024-09-30 22:00:00,0.728498,0.549680,17.445296,0.002801,188.382440,0.0,188.380107,205.434317,0.0,0.973773,65200.101303,60127.031743,9.640922,0.002561,60680.182306,0.000000,190.450690,190.195472,188.382440,191.467865,0.0,186.218501,0.851079,0.731995,NaN,NaN,22,9,274,0.0,0.000000,0.0,0.0


## 4) Model Training & Hyperparameter Tuning

We train multiple models (RandomForest, XGBoost, and CatBoost) using GridSearchCV with a time-series split. We then compare the cross-validation scores (using negative MAE) to select the best model.

In [6]:
# Define training features and target
X_train = df_train[feature_cols].dropna()
y_train = df_train.loc[X_train.index, target_col]

print('X_train shape:', X_train.shape)
print('y_train shape:', y_train.shape)

# Set up time-series cross-validation
tscv = TimeSeriesSplit(n_splits=3)

### RandomForestRegressor tuning
rf = RandomForestRegressor(random_state=42)
rf_param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [5, 10, 15]
}
grid_rf = GridSearchCV(rf, rf_param_grid, scoring='neg_mean_absolute_error', cv=tscv, verbose=1)
grid_rf.fit(X_train, y_train)
print('\nBest RF Params:', grid_rf.best_params_)
print('Best RF Score (neg MAE):', grid_rf.best_score_)

### XGBRegressor tuning
xgb = XGBRegressor(random_state=42, objective='reg:squarederror')
xgb_param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1]
}
grid_xgb = GridSearchCV(xgb, xgb_param_grid, scoring='neg_mean_absolute_error', cv=tscv, verbose=1)
grid_xgb.fit(X_train, y_train)
print('\nBest XGB Params:', grid_xgb.best_params_)
print('Best XGB Score (neg MAE):', grid_xgb.best_score_)

# ### CatBoostRegressor tuning
# cat = CatBoostRegressor(random_state=42, silent=True)
# cat_param_grid = {
#     'iterations': [200, 500],
#     'depth': [4, 6, 8],
#     'learning_rate': [0.01, 0.1]
# }
# grid_cat = GridSearchCV(cat, cat_param_grid, scoring='neg_mean_absolute_error', cv=tscv, verbose=1)
# grid_cat.fit(X_train, y_train)
# print('\nBest CatBoost Params:', grid_cat.best_params_)
# print('Best CatBoost Score (neg MAE):', grid_cat.best_score_)

# Compare cross-validation scores and select the best model
model_scores = {
    'RandomForest': grid_rf.best_score_,
    'XGBoost': grid_xgb.best_score_,
    #'CatBoost': grid_cat.best_score_
}
best_model_name = max(model_scores, key=model_scores.get)
if best_model_name == 'RandomForest':
    best_model = grid_rf.best_estimator_
elif best_model_name == 'XGBoost':
    best_model = grid_xgb.best_estimator_
else:
    best_model = grid_cat.best_estimator_

model_name = best_model_name
print(f"Selected Best Model: {model_name}")

# Evaluate on training data
y_train_pred = best_model.predict(X_train)
train_mae = mean_absolute_error(y_train, y_train_pred)
print(f"Training MAE ({model_name}): {train_mae:.4f}")

X_train shape: (9055, 6)
y_train shape: (9055,)
Fitting 3 folds for each of 6 candidates, totalling 18 fits


ValueError: 
All the 18 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
18 fits failed with the following error:
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.12/site-packages/sklearn/model_selection/_validation.py", line 895, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/opt/anaconda3/lib/python3.12/site-packages/sklearn/base.py", line 1474, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.12/site-packages/sklearn/ensemble/_forest.py", line 363, in fit
    X, y = self._validate_data(
           ^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.12/site-packages/sklearn/base.py", line 650, in _validate_data
    X, y = check_X_y(X,

## 5) Prediction on September Data & Exporting Predictions

We apply the best model on the September test set (`df_sep`), compute MAE (if actual target values exist), and export the predictions to a CSV file.

In [7]:
print(df_sep.shape)
print(df_sep.head())

(720, 34)
                datetime  W_10uheightAboveGround_10  \
9695 2024-09-01 00:00:00                  -1.742699   
9696 2024-09-01 01:00:00                  -1.229084   
9697 2024-09-01 02:00:00                  -1.780521   
9698 2024-09-01 03:00:00                  -1.795885   
9699 2024-09-01 04:00:00                  -1.619519   

      W_10vheightAboveGround_10  W_2rheightAboveGround_2  \
9695                   0.636833                43.869788   
9696                  -0.223614                48.359386   
9697                   0.135397                50.219362   
9698                  -0.083438                50.540048   
9699                  -0.565965                49.898677   

      W_2shheightAboveGround_2  W_2theightAboveGround_2  W_SUNSDsurface_0  \
9695                  0.007736               189.323875       1637.420591   
9696                  0.008078               188.763364          0.000000   
9697                  0.008205               188.542799          0.

In [8]:
# Ensure required feature columns are present in df_sep
df_sep['dswrfsurface_0'] = df_sep['W_dswrfsurface_0']
df_sep['tccatmosphere_0'] = df_sep['W_tccatmosphere_0']
df_sep['dswrf_sq'] = df_sep['dswrfsurface_0'] ** 2
df_sep['dswrf_sqrt'] = np.sqrt(np.maximum(df_sep['dswrfsurface_0'], 0))

# Drop rows with missing values in feature columns
df_sep.dropna(subset=feature_cols, inplace=True)

# Build the feature matrix for September data
X_sep = df_sep[feature_cols]

# Predict using the best model
y_sep_pred = best_model.predict(X_sep)

# Store predictions in df_sep
df_sep['pv_generation_pred'] = y_sep_pred

# Compute MAE if actual 'pv_generation' exists
if 'pv_generation' in df_sep.columns:
    actual_mask = df_sep['pv_generation'].notna()
    if actual_mask.any():
        mae_sep = mean_absolute_error(
            df_sep.loc[actual_mask, 'pv_generation'],
            df_sep.loc[actual_mask, 'pv_generation_pred']
        )
        print(f"September MAE ({model_name}): {mae_sep:.4f}")
    else:
        print("No non-null 'pv_generation' in September data for evaluation.")
else:
    print("No 'pv_generation' column in df_sep; cannot compute MAE for September.")

# Export predictions to CSV
export_cols = ['datetime', 'pv_generation_pred']
df_sep[export_cols].to_csv('september_predictions.csv', index=False, float_format='%.4f')
print("Predictions exported to 'september_predictions.csv'")

September MAE (RandomForest): 10.8967
Predictions exported to 'september_predictions.csv'


## Wrap‐Up

In this notebook, we:
1. **Merged** data from previous stages (EDA, cleaning, and feature engineering) using **cleanDatav3.csv**.
2. Performed **further feature engineering** (created time features, polynomial and interaction terms).
3. Checked for multicollinearity using **VIF**.
4. Split the data using a **time‐based split** (training up to 2024-08-31 and reserving September 2024 for prediction).
5. Tuned multiple models (RandomForest, XGBoost, and CatBoost) using **GridSearchCV** with a time-series split and selected the model with the best MAE.
6. Generated predictions for September 2024, computed the MAE (if actual values exist), and exported the predictions to a CSV file.

## Conclusion

We achieved a training MAE of approximately the value shown, while our evaluation MAE on September data is around 11.0. Further improvements can include:
1. **Refining Feature Engineering:** Adding more advanced polynomial terms or domain-driven transformations.
2. **Adding Regularization:** Tuning regularization parameters (e.g., `reg_lambda`, `reg_alpha`) in XGBoost or exploring CatBoost.
3. **Considering External Data:** Incorporating additional inputs such as holidays, day-of-week indicators, or enhanced weather data.
4. **Iterative Improvement:** Continuously monitoring MAE on the September test set and refining the model accordingly.
5. **Integration with Battery Logic:** Once the solar forecast is optimized, integrating it with battery optimization and CO₂ reduction calculations.

Exporting predictions enables easy submission via the official challenge forms. Keep iterating to further reduce the MAE!